# EventLens model experiment playground

This is the broad experiment log. It tries several model families, regularization strengths, feature sets, horizons, and a log-variance transformation. The goal is not to cherry-pick a winner: every attempt is written to CSV, with diagnostics and an interpretation report saved for the final report.


## 1. Get data and build the same point-in-time modelling table

The experiment reuses the cleaned SPY and prediction-market data. It never uses `final_outcome` as an input. Forecast origins are chronological and multi-day origins are spaced by the horizon so targets do not overlap.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'datasets').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'models'))
from playground_experiments import build_data, diagnostics, run_experiments, write_report, OUTPUT_DIR

data, probability = build_data()
diagnostic = diagnostics(data, probability)
diagnostic


## 2. Experiment dimensions

Feature sets: **history-only**, **event-only** (return target only), and **history-plus-event**. Models: Ridge with three penalties, Elastic Net, RBF SVR, Random Forest, Extra Trees, Gradient Boosting, and Histogram Gradient Boosting. Variance is tried both on the original scale and on a log-transformed target, because variance must be non-negative and the raw linear setup can produce invalid negative values.


In [ ]:
results = run_experiments(data)
results.head()


## 3. Compare models

For each target and horizon, lower RMSE is better. Compare history-plus-event only against the best history-only model at the same horizon. The event-only model is deliberately included to test whether event information has signal without recent SPY history.


In [ ]:
best = (results.sort_values(['target', 'horizon', 'rmse'])
         .groupby(['target', 'horizon'], as_index=False)
         .head(5))
best


## 4. Diagnose the result

A feature model winning once is not enough. We also inspect the largest feature correlation, the feature-matrix condition number, the number of unique events, and the number of held-out cases. High collinearity, few independent events, or a 15-row test set can make an apparent improvement unreliable.


In [ ]:
results.to_csv(OUTPUT_DIR / 'playground_experiment_results.csv', index=False)
import pandas as pd
pd.DataFrame([diagnostic]).to_csv(OUTPUT_DIR / 'playground_diagnostics.csv', index=False)
write_report(results, diagnostic)
print('Saved full experiment log, diagnostics, and report.')


## 5. How to interpret the experiment

- If history-plus-event beats history-only across several models and horizons, the evidence for incremental event information is stronger.
- If only one flexible model wins on the 20-day test, it may be chance or overfitting.
- If Ridge and tree models agree, the result is more credible.
- If log-variance models win, the raw variance scale was probably a poor modelling choice.
- If event-only is weak but combined is better, events may add small information on top of market history.
- If combined is worse across families, likely explanations include noisy features, collinearity, repeated event snapshots, too few independent events, or event timing that does not match the target.

The saved Markdown report records the actual winners and the percentage change from history-only to history-plus-event.
